# ECDS Shock Index Overview

This notebook introduces the initial data sources and formula used to compute an ECDS Shock Index for Medicare Advantage Stars measures.

## Data Sources

- NCQA HEDIS ECDS results (example extract in `data/raw/example_ncqa_ecds.csv`)
- CMS Stars measure definitions and weights (example extract in `data/raw/example_cms_measure_weights.csv`)

## Index Formula

The measure-level index is **hazard-only**:

\[\text{ShockIndex} = \alpha\cdot CCS + \beta\cdot EAV + \gamma\cdot CPR\]

Default weights in this repository (neutral renormalization of the previous hazard weights — see [`ADR-0002`](../docs/decisions/0002-remove-wm-from-measure-composite.md)):
- \alpha = 0.4375
- \beta = 0.3125
- \gamma = 0.25

All components are scores in [0, 1] (higher = more risk): CCS is a completeness *gap*, EAV is **log-scale** deviation from a stable baseline (`eav(r) == eav(1/r)`, [`ADR-0003`](../docs/decisions/0003-log-symmetric-eav.md)), and CPR is anchored to the CMS cut-point guardrail (5 pp on 0–100 scale; not applied to improvement measures or measures in the program 3 years or less). WM (Stars weight / 5) is an **exposure** indicator: it is reported per measure and enters the analysis exactly once, as the aggregation weight in the contract-level rollup. See [`../docs/literature-evaluation.md`](../docs/literature-evaluation.md) for the evidence base.


In [1]:
from ecds_shock_index import ShockIndexCalculator, ccs_score, eav_score, cpr_score, wm_score, classify_risk

calculator = ShockIndexCalculator()

# Single-measure example (hazard components only; wm is reported, not composited)
ccs = ccs_score(0.82, 0.90)
eav = eav_score(1.15)
cpr = cpr_score(0.20)
wm = wm_score(3, max_weight=5)

index = calculator.calculate(ccs=ccs, eav=eav, cpr=cpr)
tier = classify_risk(index)
print(f"Shock Index: {index:.4f}  ({tier} risk)")

Shock Index: 0.3393  (moderate risk)


## Batch Scoring

Load the example CSVs, score every measure, and aggregate to a contract-level summary.

In [2]:
from ecds_shock_index.data_loader import load_ncqa_ecds, load_cms_measure_weights, merge_ecds_and_weights

# Load and merge data
ecds = load_ncqa_ecds("../data/raw/example_ncqa_ecds.csv")
weights = load_cms_measure_weights("../data/raw/example_cms_measure_weights.csv")
merged = merge_ecds_and_weights(ecds, weights)

# Score all measures at once
scored = calculator.score_dataframe(merged)
scored[["measure_id", "ccs", "eav", "cpr", "wm", "shock_index", "risk_tier"]]

,measure_id,ccs,eav,cpr,wm,shock_index,risk_tier
0,COL,0.060,0.042644,0.320,0.2,0.119576,low
1,BCS,0.300,0.536053,0.816,0.2,0.502767,high
2,A1C,0.135,0.137504,0.800,0.6,0.302032,moderate


## Contract-Level Aggregation

Combine individual measure scores into a single contract-level risk summary.

In [3]:
# Contract-level summary (weighted by CMS measure weight)
summary = calculator.aggregate_contract(scored)
for key, val in summary.items():
    print(f"  {key}: {val}")

  weighted_shock_index: 0.3057
  mean_shock_index: 0.3081
  max_shock_index: 0.5028
  measure_count: 3
  risk_tier: moderate
